# Pupil extraction

This notebook runs the pupil pipeline in three stages: strict clock/count validation, a sparse session-wide segmentation QC montage, and the full streaming extraction. The full pass is intentionally not started if the video, `cameraFrameSync`, and `frametimes.csv` counts disagree.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse

# Works when the notebook is opened from either the repository root or analysis/.
REPO = Path.cwd().resolve()
if not (REPO / 'analysis').is_dir():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from analysis.session.pupil import (
    PupilConfig, count_csv_frames, count_video_frames,
    fit_ellipse_ransac, iter_gray_frames, pupil_from_round,
    segment_bright_pupil, validate_alignment_counts,
)
from analysis.session.respiration import find_behavior_sync
from analysis.session.sync import frame_onset_samples, open_sync

## Configuration

Set the four paths. `ROI` is `(y0, y1, x0, x1)` and should tightly contain the eye; leaving it as `None` thresholds the whole frame. Start with the sparse QC below and adjust the ROI before running the full extraction.

In [ ]:
VIDEO_PATH = Path('/path/to/behavior_camera.mp4')
FRAMETIMES_PATH = Path('/path/to/frametimes.csv')
ROUND_PATH = Path('/path/to/group_processed_round.h5')
SYNC_PATH = None  # Path('/path/to/behavior_sync.h5'), or auto-discover
OUT_DIR = ROUND_PATH.parent / 'aux'

ROI = None  # Example: (60, 330, 80, 370)
CONFIG = PupilConfig(
    roi=ROI,
    ransac_residual_px=2.0,
    ransac_trials=200,
    min_inlier_fraction=0.55,
    max_residual_px=3.0,
    min_axis_ratio=0.25,
    max_diameter_rate_px_s=150.0,
    max_bad_fraction=0.20,
)

N_QC_FRAMES = 16

## 1. Alignment preflight

This performs one streaming decode solely to obtain the exact decoded frame count. A mismatch raises immediately, before segmentation or output writing.

In [ ]:
for path in (VIDEO_PATH, FRAMETIMES_PATH, ROUND_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

sync_path = Path(SYNC_PATH) if SYNC_PATH is not None else find_behavior_sync(ROUND_PATH.parents[2])
sync = open_sync(sync_path)
camera_samples = frame_onset_samples(sync, channel='cameraFrameSync')
n_camera = validate_alignment_counts(VIDEO_PATH, FRAMETIMES_PATH, camera_samples)
camera_time_s = camera_samples / sync.rate_hz
duration_min = (camera_time_s[-1] - camera_time_s[0]) / 60

print(f'PASS: {n_camera:,} decoded frames = camera pulses = CSV rows')
print(f'Sync: {sync_path}')
print(f'Camera duration: {duration_min:.2f} min')
print(f'Median camera rate: {1 / np.median(np.diff(camera_time_s)):.3f} Hz')

## 2. Sparse segmentation QC

The video is streamed again, but only evenly spaced frames are segmented. Cyan is the filled largest-component mask and magenta is the RANSAC ellipse. The chosen Otsu threshold, inlier fraction, residual, and equivalent diameter are printed above each frame. This montage is saved before the full extraction begins.

In [ ]:
sample_indices = set(np.linspace(0, n_camera - 1, N_QC_FRAMES, dtype=int))
samples = []
for i, frame in enumerate(iter_gray_frames(VIDEO_PATH)):
    if i not in sample_indices:
        continue
    mask, threshold = segment_bright_pupil(frame, roi=CONFIG.roi)
    fit = fit_ellipse_ransac(
        mask, residual_px=CONFIG.ransac_residual_px,
        max_trials=CONFIG.ransac_trials, random_seed=CONFIG.random_seed + i,
    )
    samples.append((i, frame, mask, threshold, fit))

n_col = 4
n_row = int(np.ceil(len(samples) / n_col))
fig, axes = plt.subplots(n_row, n_col, figsize=(14, 3.5 * n_row), squeeze=False)
for ax, item in zip(axes.flat, samples):
    i, frame, mask, threshold, fit = item
    ax.imshow(frame, cmap='gray')
    ax.contour(mask, levels=[0.5], colors='cyan', linewidths=0.8)
    if fit is not None:
        ax.add_patch(Ellipse(
            (fit['x'], fit['y']), 2 * fit['major'], 2 * fit['minor'],
            angle=np.degrees(fit['theta']), fill=False,
            edgecolor='magenta', linewidth=1.2,
        ))
        detail = f"in={fit['inlier_fraction']:.2f}  res={fit['residual']:.2f}  d={fit['diameter']:.1f}"
    else:
        detail = 'FIT FAILED'
    ax.set_title(f'frame {i:,}  t={threshold:.0f}\n{detail}', fontsize=9)
    ax.axis('off')
for ax in axes.flat[len(samples):]:
    ax.axis('off')
fig.suptitle('Sparse pupil segmentation preflight: cyan mask, magenta RANSAC ellipse')
OUT_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_QC_PATH = OUT_DIR / f'{ROUND_PATH.stem}_pupil_preflight_qc.png'
fig.savefig(INTERMEDIATE_QC_PATH, dpi=160, bbox_inches='tight')
plt.show()
print(f'Saved: {INTERMEDIATE_QC_PATH}')

Inspect the montage now. If the cyan component is not consistently the pupil, tighten `ROI`. If thresholds drift onto sclera or background, correct the ROI or illumination before proceeding. Re-run the configuration and sparse-QC cells after changes.

## 3. Full streaming extraction

This is the long pass. It repeats the strict count preflight by design, processes one frame at a time, aligns camera frame `i` to camera pulse `i`, and samples onto the 2p frame grid using the shared clock.

In [ ]:
report = pupil_from_round(
    VIDEO_PATH, FRAMETIMES_PATH, ROUND_PATH,
    sync_path=sync_path, out_dir=OUT_DIR, config=CONFIG, save=True,
)

print(f"HDF5: {report['h5']}")
print(f"QC figure: {report['figure']}")
print(f"Trials: {report['n_trial']}, frames/trial: {report['n_frame']}")
print(f"Flagged trials: {int(report['flagged'].sum())}/{report['n_trial']}")
print(f"Median blink fraction: {np.median(report['blink_fraction']):.2%}")
print(f"Median clipping fraction: {np.median(report['clipped_fraction']):.2%}")

## 4. Final QC outputs

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(report['figure'])))

worst_trials = np.argsort(report['masked_fraction'])[::-1][:10]
print('Worst trials by masked fraction:')
for row in worst_trials:
    print(
        f"  trial row {row:3d}, acq_id={int(report['acq_id'][row])}, "
        f"masked={report['masked_fraction'][row]:.1%}, "
        f"blink={report['blink_fraction'][row]:.1%}, "
        f"clipped={report['clipped_fraction'][row]:.1%}"
    )